# Large-scale post-convergence boundary test — cleaned final protocol

This is a structural cleanup of `route_viability_pilot(1)(1).ipynb`. Exact duplicate cells, exploratory pair-task versions, the obsolete load-32 switch-at-3000 run, and repeated efficiency cells were removed. The final selective-recall task, position-wise trunk, strictly-earlier retrieval, load-8 switch at 3000, load-32 switch at 5000, three seeds, and λ=0.1 are encoded directly.

**Important:** the supplied final position-wise implementation reports ~19.77M parameters, while the current paper text says 30.17M. This notebook does not alter widths to force the paper number. See `docs/REPRODUCIBILITY_AUDIT.md`.


In [ ]:
import hashlib, json, math, os, pathlib, random, time
from dataclasses import dataclass, asdict
from contextlib import nullcontext

PRESET = "full"          # "smoke" for a short pipeline check
USE_DRIVE = False         # set True in Colab if you want Drive persistence
DRIVE_DIR = "/content/drive/MyDrive/route_pilot"

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    WORKDIR = DRIVE_DIR
else:
    WORKDIR = os.environ.get("PILOT_WORKDIR", str(pathlib.Path.cwd() / "route_pilot"))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib as mpl
import matplotlib.pyplot as plt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IS_CUDA = DEVICE.type == "cuda"
GPU_NAME = torch.cuda.get_device_name(0) if IS_CUDA else "cpu"
AMP_DTYPE = (torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16) if IS_CUDA else torch.bfloat16

def amp():
    return torch.autocast(device_type=DEVICE.type, dtype=AMP_DTYPE, enabled=IS_CUDA)

def sync():
    if IS_CUDA:
        torch.cuda.synchronize()

def P(full, smoke):
    return full if PRESET == "full" else smoke

SIZE = "M"
LOADS = P([2, 4, 8, 16, 32], [2, 4])
SEEDS = P([0], [0])
STEPS = P(8000, 200)
CURRICULUM = 0.0
LOG = P(500, 50)
N_EVAL = P(8, 2)
CKPT_MINUTES = P(10.0, 2.0)
VIABLE_ACC = 0.95
TIME_BUDGET_H = P(3.0, 0.2)
MODELS = ["recurrent_only", "attention_only", "hybrid"]
FORMAT_VERSION = 3
IGNORE = -100

@dataclass
class Cfg:
    d: int = 512
    n_layers: int = 8
    n_heads: int = 8
    mlp_mult: int = 4
    lr: float = 6e-4
    warmup: int = 300
    weight_decay: float = 0.0
    grad_clip: float = 1.0
    batch: int = 64
    n_query: int = 4

SIZES = {"S": dict(d=256, n_layers=6, n_heads=4, lr=1e-3),
         "M": dict(d=512, n_layers=8, n_heads=8, lr=6e-4)}
CFG = Cfg(**SIZES[SIZE])
if PRESET == "smoke":
    CFG.warmup = 20

TAG = f"{SIZE}_{PRESET}"
WORK = pathlib.Path(WORKDIR).expanduser()
RESDIR, CKPTDIR, FIGDIR, TABDIR = (WORK / f"{n}_{TAG}" for n in ("results", "checkpoints", "figures", "tables"))
for _d in (RESDIR, CKPTDIR, FIGDIR, TABDIR):
    _d.mkdir(parents=True, exist_ok=True)
T_START = time.time()

def set_seed(s):
    torch.manual_seed(s); np.random.seed(s); random.seed(s)

print(f"{GPU_NAME} | torch {torch.__version__} | loads {LOADS} | {STEPS} steps | batch {CFG.batch}")
print("work dir:", WORK)


In [ ]:
def keyof(spec):
    return hashlib.md5(json.dumps(spec, sort_keys=True, default=str).encode()).hexdigest()[:16]


class Records:
    def __init__(self, path):
        self.path, self.recs = path, {}
        if path.exists():
            for line in path.read_text().splitlines():
                try:
                    r = json.loads(line); self.recs[r["_key"]] = r
                except (json.JSONDecodeError, KeyError):
                    pass

    def get(self, spec):
        return self.recs.get(keyof(spec))

    def put(self, spec, payload):
        r = {"_key": keyof(spec), "_spec": spec, "_t": time.time(), **payload}
        self.recs[r["_key"]] = r
        with open(self.path, "a") as f:
            f.write(json.dumps(r, default=float) + "\n"); f.flush(); os.fsync(f.fileno())
        return r

    def frame(self):
        return pd.DataFrame([{**r["_spec"], **{k: v for k, v in r.items()
                                               if not k.startswith("_") and k != "trace"}}
                             for r in self.recs.values()])


def ckpt_save(name, obj):
    p = CKPTDIR / f"{name}.pt"
    torch.save(obj, p.with_suffix(".tmp")); os.replace(p.with_suffix(".tmp"), p)


def ckpt_load(name):
    p = CKPTDIR / f"{name}.pt"
    if p.exists():
        try:
            return torch.load(p, map_location=DEVICE, weights_only=False)
        except Exception as e:
            print(f"  checkpoint unreadable ({type(e).__name__})")
    return None


def ckpt_delete(name):
    for suf in (".pt", ".tmp"):
        (CKPTDIR / f"{name}{suf}").unlink(missing_ok=True)


REC = Records(RESDIR / "records.jsonl")
print(f"{len(REC.recs)} finished run(s) on disk")


## Final selective-recall task

In [ ]:
# Back to the paper's selective recall: L items from a small alphabet, then a query naming a
# POSITION. Recurrence stores an ordered list; retrieval matches the position and reads the item.
# Both are learnable, which 512-way key-value binding was not. Also drops the curriculum, which
# made the hybrid commit to recurrence at load 1 and stall.
FORMAT_VERSION = 3
CURRICULUM = 0.0
N_ITEMS, MAXL = 16, max(LOADS)
PAD, SEP, QRY = 0, 1, 2
POS0 = 3
ITEM0 = POS0 + MAXL
VOCAB = ITEM0 + N_ITEMS
CHANCE = 1.0 / N_ITEMS


def make_batch(B, load, gen):
    dev = gen.device
    nq = min(CFG.n_query, load)
    T = 1 + load + 1 + 2 * nq
    items = torch.randint(0, N_ITEMS, (B, load), generator=gen, device=dev)
    qidx = torch.rand(B, load, generator=gen, device=dev).argsort(-1)[:, :nq]
    x_key = torch.full((B, T), PAD, dtype=torch.long, device=dev)
    x_val = torch.full((B, T), PAD, dtype=torch.long, device=dev)
    y = torch.full((B, T), IGNORE, dtype=torch.long, device=dev)
    x_key[:, 0] = SEP
    x_val[:, 1:1 + load] = ITEM0 + items
    p = 1 + load
    x_key[:, p] = SEP
    qpos = p + 1 + 2 * torch.arange(nq, device=dev)
    x_key[:, qpos] = QRY
    x_key[:, qpos + 1] = POS0 + qidx                      # the query names a position
    y[:, qpos + 1] = ITEM0 + items.gather(1, qidx)        # the item at that position
    return (x_key.to(DEVICE, non_blocking=True), x_val.to(DEVICE, non_blocking=True),
            y.to(DEVICE, non_blocking=True))


def show(k, v):
    out = []
    for a, b in zip(k.tolist(), v.tolist()):
        if b >= ITEM0: out.append(f"x{b - ITEM0}")
        elif a == SEP: out.append("|")
        elif a == QRY: out.append("?")
        elif a >= POS0: out.append(f"Q{a - POS0}")
        else: out.append("_")
    return " ".join(out)


_g = torch.Generator(device="cpu"); _g.manual_seed(0)
_k, _v, _y = make_batch(1, 4, _g)
print("example (4 items):", show(_k[0].cpu(), _v[0].cpu()))
print("targets           :", [(i, f"x{int(t) - ITEM0}") for i, t in enumerate(_y[0]) if t != IGNORE])
print(f"{N_ITEMS} item symbols, chance {CHANCE:.4f} | vocab {VOCAB}")


def eval_batches(load, n=N_EVAL, seed=4321):
    g = torch.Generator(device='cpu'); g.manual_seed(seed + load)
    return [make_batch(64, load, g) for _ in range(n)]


## Final model

In [ ]:
# Final architecture used by the FORMAT_VERSION=3 selective-recall runs.
# The trunk is strictly position-wise: all cross-position state is in the single GRU.
class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__(); self.w, self.eps = nn.Parameter(torch.ones(d)), eps
    def forward(self, x):
        return self.w * x * torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps).to(x.dtype)

class TrunkBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.n = RMSNorm(cfg.d)
        self.mlp = nn.Sequential(nn.Linear(cfg.d, cfg.mlp_mult * cfg.d, bias=False), nn.GELU(),
                                 nn.Linear(cfg.mlp_mult * cfg.d, cfg.d, bias=False))
    def forward(self, x):
        return x + self.mlp(self.n(x))

class DualRoute(nn.Module):
    def __init__(self, cfg, recurrent=True, retrieval=True):
        super().__init__()
        self.cfg, self.recurrent, self.retrieval = cfg, recurrent, retrieval
        self.emb_key = nn.Embedding(VOCAB, cfg.d)
        self.emb_val = nn.Embedding(VOCAB, cfg.d)
        self.trunk = nn.ModuleList([TrunkBlock(cfg) for _ in range(cfg.n_layers)])
        self.norm = RMSNorm(cfg.d)
        if recurrent:
            self.gru = nn.GRU(cfg.d, cfg.d, batch_first=True)
            self.proj_h = nn.Linear(cfg.d, cfg.d)
            self.w_h = nn.Linear(cfg.d, VOCAB, bias=False)
        if retrieval:
            self.q = nn.Linear(cfg.d, cfg.d, bias=False)
            self.k = nn.Linear(cfg.d, cfg.d, bias=False)
            self.v = nn.Linear(cfg.d, cfg.d, bias=False)
            self.proj_c = nn.Linear(cfg.d, cfg.d)
            self.w_c = nn.Linear(cfg.d, VOCAB, bias=False)
        for m in self.modules():
            if isinstance(m, (nn.Linear, nn.Embedding)):
                nn.init.normal_(m.weight, std=0.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.zeros_(m.bias)

    def trunk_states(self, x_key, x_val):
        h = self.emb_key(x_key) + self.emb_val(x_val)
        for blk in self.trunk:
            h = blk(h)
        return self.norm(h)

    def routes(self, x_key, x_val, qpos):
        S = self.trunk_states(x_key, x_val)
        B, T, D = S.shape
        idx = qpos[None, :, None].expand(B, -1, D)
        h_rec = c = None
        if self.recurrent:
            g_all, _ = self.gru(S)
            h_rec = g_all.gather(1, idx)
        else:
            g_all = S
        if self.retrieval:
            q = self.q(g_all.gather(1, idx))
            k, v = self.k(g_all), self.v(g_all)
            pos = torch.arange(T, device=S.device)
            allowed = pos[None, :] < qpos[:, None]  # strictly earlier states only
            mask = torch.zeros(len(qpos), T, device=S.device).masked_fill(~allowed, float('-inf'))
            c = F.scaled_dot_product_attention(q.unsqueeze(1), k.unsqueeze(1), v.unsqueeze(1),
                                               attn_mask=mask[None, None]).squeeze(1)
        return h_rec, c

    def logits_from(self, h_rec, c):
        lh = self.w_h(F.gelu(self.proj_h(h_rec))) if (self.recurrent and h_rec is not None) else None
        lc = self.w_c(F.gelu(self.proj_c(c))) if (self.retrieval and c is not None) else None
        if lh is None and lc is None:
            raise RuntimeError('logits_from called with no route')
        out = lh if lc is None else (lc if lh is None else lh + lc)
        return out, lh, lc

    def forward(self, x_key, x_val, qpos, drop=None, perm=None):
        h_rec, c = self.routes(x_key, x_val, qpos)
        ref = h_rec if h_rec is not None else c
        if drop == 'recurrent':
            h_rec = None
        elif perm is not None and perm[0] == 'recurrent' and h_rec is not None:
            h_rec = h_rec[perm[1]]
        if drop == 'retrieval':
            c = None
        elif perm is not None and perm[0] == 'retrieval' and c is not None:
            c = c[perm[1]]
        if h_rec is None and c is None:
            return torch.zeros(ref.shape[0], ref.shape[1], VOCAB, device=ref.device), None, None
        return self.logits_from(h_rec, c)

def build(kind, seed):
    set_seed(seed)
    return DualRoute(CFG, recurrent=kind != 'attention_only', retrieval=kind != 'recurrent_only').to(DEVICE)

for _k in MODELS:
    _m = build(_k, 0)
    _n = sum(p.numel() for p in _m.parameters())
    print(f"{_k:<16}{_n/1e6:6.2f}M parameters")
del _m


## Evaluation and viability training

In [ ]:
def qpos_of(y):
    pos = (y[0] != IGNORE).nonzero(as_tuple=True)[0]
    return pos


@torch.no_grad()
def evaluate(model, batches):
    was = model.training
    model.eval()
    tot = {k: 0.0 for k in ("clean", "rec_res", "ret_res", "no_rec", "no_ret")}
    n = 0
    for x_key, x_val, y in batches:
        qpos = qpos_of(y)
        tgt = y[:, qpos]
        B = x_key.shape[0]
        perm = torch.roll(torch.arange(B, device=DEVICE), 1)

        def hit(**kw):
            with amp():
                lg = model(x_key, x_val, qpos, **kw)[0]
            return float((lg.argmax(-1) == tgt).sum())

        tot["clean"] += hit()
        if model.recurrent:
            tot["rec_res"] += hit(perm=("recurrent", perm))
            tot["no_rec"] += hit(drop="recurrent")
        if model.retrieval:
            tot["ret_res"] += hit(perm=("retrieval", perm))
            tot["no_ret"] += hit(drop="retrieval")
        n += tgt.numel()
    m = {k: v / n for k, v in tot.items()}
    acc = m["clean"]
    head = max(acc - CHANCE, 1e-9)
    out = {"acc": acc,
           "D_rec": float(np.clip((acc - m["rec_res"]) / head, 0, 1)) if model.recurrent else float("nan"),
           "D_ret": float(np.clip((acc - m["ret_res"]) / head, 0, 1)) if model.retrieval else float("nan"),
           "acc_no_rec": m["no_rec"] if model.recurrent else acc,
           "acc_no_ret": m["no_ret"] if model.retrieval else acc}
    if was:
        model.train()
    return out


def train_run(key, kind, load, seed):
    model = build(kind, seed)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG.lr, betas=(0.9, 0.95), weight_decay=CFG.weight_decay)
    scaler = torch.amp.GradScaler("cuda", enabled=IS_CUDA and AMP_DTYPE == torch.float16)
    batches = eval_batches(load)
    start, trace, rt, rs = 0, [], 0.0, 0
    ck = ckpt_load(f"run_{key}")
    if ck is not None:
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
        if ck["scaler"]:
            scaler.load_state_dict(ck["scaler"])
        start, trace, rt, rs = ck["step"], ck["trace"], ck["rt"], ck["rs"]
        print(f"    resumed at step {start}/{STEPS}")
    rng = np.random.default_rng([seed, int(key[:8], 16), start])
    gen = torch.Generator(device=DEVICE); gen.manual_seed(int(rng.integers(2 ** 62)))
    model.train()
    last_ck, losses = time.time(), []
    ramp = max(1, int(CURRICULUM * STEPS))
    for step in range(start + 1, STEPS + 1):
        sync(); t0 = time.time()
        for g in opt.param_groups:
            g["lr"] = CFG.lr * min(1.0, step / CFG.warmup)
        L = load if step > ramp else max(1, int(round(1 + (load - 1) * step / ramp)))
        x_key, x_val, y = make_batch(CFG.batch, L, gen)
        qpos = qpos_of(y)
        with amp():
            lg = model(x_key, x_val, qpos)[0]
        loss = F.cross_entropy(lg.float().reshape(-1, VOCAB), y[:, qpos].reshape(-1))
        opt.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip)
        scaler.step(opt); scaler.update()
        sync(); rt += time.time() - t0; rs += 1
        losses.append(float(loss.detach()))
        if step == start + 20:
            print(f"    {1000 * rt / rs:.0f} ms/step, about {rt / rs * (STEPS - step) / 60:.0f} min left", flush=True)
        if step % LOG == 0 or step == STEPS:
            ev = evaluate(model, batches)
            trace.append({"step": step, "loss": float(np.mean(losses)), "curriculum_load": L, **ev})
            print(f"    step {step:>5}  loss {np.mean(losses):.3f}  acc {ev['acc']:.3f}"
                  + (f"  D_rec {ev['D_rec']:.2f} D_ret {ev['D_ret']:.2f}" if kind == "hybrid" else ""),
                  flush=True)
            losses = []
        if step < STEPS and time.time() - last_ck > 60 * CKPT_MINUTES:
            ckpt_save(f"run_{key}", {"model": model.state_dict(), "opt": opt.state_dict(),
                                     "scaler": scaler.state_dict(), "step": step, "trace": trace,
                                     "rt": rt, "rs": rs})
            last_ck = time.time()
    out = {**evaluate(model, batches), "trace": trace, "ms_per_step": 1000 * rt / max(rs, 1),
           "params": sum(p.numel() for p in model.parameters())}
    ckpt_delete(f"run_{key}")
    del model, opt
    if IS_CUDA:
        torch.cuda.empty_cache()
    return out


In [ ]:
# One short timing probe, then the runs. Loads are done in order and the notebook stops adding
# loads once the time budget is spent, so a partial result is still a usable boundary.
_g = torch.Generator(device=DEVICE); _g.manual_seed(0)
_m = build("hybrid", 0)
_opt = torch.optim.AdamW(_m.parameters(), lr=CFG.lr)
_xk, _xv, _y = make_batch(CFG.batch, max(LOADS), _g)
_qp = qpos_of(_y)
for _ in range(3):
    with amp():
        _l = _m(_xk, _xv, _qp)[0]
    F.cross_entropy(_l.float().reshape(-1, VOCAB), _y[:, _qp].reshape(-1)).backward()
    _opt.zero_grad(set_to_none=True)
sync(); _t0 = time.time()
with amp():
    _l = _m(_xk, _xv, _qp)[0]
F.cross_entropy(_l.float().reshape(-1, VOCAB), _y[:, _qp].reshape(-1)).backward()
sync()
MS_PER_STEP = 1000 * (time.time() - _t0)
del _m, _opt, _l
if IS_CUDA:
    torch.cuda.empty_cache()
_per_run_min = MS_PER_STEP * STEPS / 60000
print(f"about {MS_PER_STEP:.0f} ms/step -> {_per_run_min:.0f} min per run, "
      f"{_per_run_min * len(MODELS) * len(LOADS) * len(SEEDS) / 60:.1f} h for all runs "
      f"(budget {TIME_BUDGET_H} h)")

print(f"\n{'model':<16}{'load':>5}{'seed':>5}{'acc':>7}{'noRec':>7}{'noRet':>7}{'D_rec':>7}{'D_ret':>7}{'min':>6}")
for load in LOADS:
    for kind in MODELS:
        for seed in SEEDS:
            spec = {"block": "run", "kind": kind, "load": load, "seed": seed, "size": SIZE,
                    "steps": STEPS, "cfg": asdict(CFG), "format": FORMAT_VERSION}
            r = REC.get(spec)
            if r is None:
                print(f"  [{kind} load {load} seed {seed}]", flush=True)
                r = REC.put(spec, train_run(keyof(spec), kind, load, seed))
            print(f"{kind:<16}{load:>5}{seed:>5}{r['acc']:>7.3f}{r['acc_no_rec']:>7.3f}"
                  f"{r['acc_no_ret']:>7.3f}{r['D_rec']:>7.2f}{r['D_ret']:>7.2f}"
                  f"{r.get('ms_per_step', float('nan')) * STEPS / 60000:>6.0f}", flush=True)


In [ ]:
R = REC.frame()
R = R[(R["size"] == SIZE) & (R["steps"] == STEPS)] if len(R) else R
lines = [f"=== ROUTE-VIABILITY PILOT | {time.strftime('%Y-%m-%d %H:%M')} ===",
         f"{GPU_NAME} | {str(AMP_DTYPE).split('.')[-1]} | size {SIZE} (d={CFG.d}, {CFG.n_layers} layers) | "
         f"{STEPS} steps, batch {CFG.batch}, curriculum over the first {int(100 * CURRICULUM)}% | "
         f"{N_ITEMS} symbols (chance {CHANCE:.4f}) | {(time.time() - T_START) / 3600:.1f} h elapsed"]

if not len(R):
    lines.append("no runs finished")
    VERDICT, BOTH = "NO RUNS", []
else:
    piv = R.pivot_table(index="load", columns="kind", values="acc", aggfunc="mean")
    for c in MODELS:
        if c not in piv:
            piv[c] = np.nan
    piv = piv[MODELS]
    hyb = R[R.kind == "hybrid"].groupby("load").agg(
        hybrid_acc=("acc", "mean"), D_rec=("D_rec", "mean"), D_ret=("D_ret", "mean"),
        acc_no_rec=("acc_no_rec", "mean"), acc_no_ret=("acc_no_ret", "mean"))
    TAB = piv.join(hyb, rsuffix="_h")
    TAB["rec_viable"] = TAB["recurrent_only"] >= VIABLE_ACC
    TAB["ret_viable"] = TAB["attention_only"] >= VIABLE_ACC
    TAB["both_viable"] = TAB.rec_viable & TAB.ret_viable
    TAB.to_csv(TABDIR / "pilot_by_load.csv")
    BOTH = [int(i) for i in TAB.index[TAB.both_viable]]
    RET_ONLY = [int(i) for i in TAB.index[TAB.ret_viable & ~TAB.rec_viable]]
    NEITHER = [int(i) for i in TAB.index[~TAB.ret_viable & ~TAB.rec_viable]]
    VERDICT = ("PRECONDITION MET" if BOTH else
               "RETRIEVAL ONLY: recurrence never viable" if RET_ONLY else
               "NEITHER ROUTE VIABLE: task still too hard")
    lines += ["", "single-route viability and the free hybrid, per load:",
              TAB.round(3).to_string(), "",
              f"both routes viable at loads: {BOTH or 'none'}",
              f"retrieval only: {RET_ONLY or 'none'} | neither: {NEITHER or 'none'}"]
    if BOTH:
        b = TAB.loc[BOTH]
        dom = b[(b.D_ret >= 0.7) & (b.D_rec <= 0.3)]
        lines.append(f"free hybrid at those loads: D_rec {list(b.D_rec.round(2))}, D_ret {list(b.D_ret.round(2))}"
                     + (f" -> retrieval-dominant at {[int(i) for i in dom.index]}" if len(dom)
                        else " -> not clearly retrieval-dominant"))
        nxt = [int(i) for i in TAB.index[TAB.ret_viable & ~TAB.rec_viable]]
        lines.append(f"suggested four-arm conditions: positive load {max(BOTH)}"
                     + (f", negative control load {min(nxt)}" if nxt else ", negative control not yet found"))
    lines.append(f"VERDICT: {VERDICT}")

    mpl.rcParams.update({"figure.dpi": 130, "savefig.dpi": 300, "savefig.bbox": "tight", "font.size": 8,
                         "axes.spines.top": False, "axes.spines.right": False, "legend.fontsize": 7})
    fig, ax = plt.subplots(1, 2, figsize=(6.6, 2.5))
    COL = {"recurrent_only": "#1b6ca8", "attention_only": "#e07a1f", "hybrid": "#3a9d5d"}
    for c in MODELS:
        ax[0].plot(TAB.index, TAB[c], "-o", ms=3, color=COL[c], label=c.replace("_", " "))
    ax[0].axhline(VIABLE_ACC, color="0.5", lw=0.8, ls="--")
    ax[0].set_xscale("log", base=2); ax[0].set_xticks(list(TAB.index)); ax[0].set_xticklabels(list(TAB.index))
    ax[0].set_ylim(-0.03, 1.03); ax[0].set_xlabel("memory load"); ax[0].set_ylabel("accuracy")
    ax[0].set_title("single-route viability"); ax[0].legend(frameon=False, loc="lower left")
    ax[1].plot(TAB.index, TAB.D_rec, "-o", ms=3, color=COL["recurrent_only"], label="$D_h$ recurrent")
    ax[1].plot(TAB.index, TAB.D_ret, "-o", ms=3, color=COL["attention_only"], label="$D_c$ retrieval")
    ax[1].set_xscale("log", base=2); ax[1].set_xticks(list(TAB.index)); ax[1].set_xticklabels(list(TAB.index))
    ax[1].set_ylim(-0.03, 1.03); ax[1].set_xlabel("memory load"); ax[1].set_ylabel("dependence")
    ax[1].set_title("which route the free hybrid uses"); ax[1].legend(frameon=False, loc="center left")
    fig.tight_layout()
    for ext in ("pdf", "png"):
        fig.savefig(FIGDIR / f"pilot_viability.{ext}")
    plt.show()

SUMMARY = "\n".join(lines)
(TABDIR / "PILOT_SUMMARY.txt").write_text(SUMMARY)
print(SUMMARY)
print(f"\nsaved {TABDIR / 'PILOT_SUMMARY.txt'}")


## Final four-arm helpers and endpoints

In [ ]:
# Final four-arm protocol: the switch is load-dependent because the free load-32 model
# is not reliably converged at step 3000. This directly encodes the final protocol instead
# of first running the obsolete 3000-step load-32 version and then rerunning it.
STEER_POS, STEER_NEG = 8, 32
SWITCH_OF = {8: 3000, 32: 5000}
POST = 8000
LAMBDA = 0.1
STEER_LOG = 500
STEER_ARMS = ['never', 'late', 'maintained', 'withdrawn']
STEER_SEEDS = [0, 1, 2]
FORMAT_STEER = 2

def price_of(step, arm, switch):
    if arm == 'never': return 0.0
    if arm == 'maintained': return LAMBDA
    if arm == 'late': return LAMBDA if step > switch else 0.0
    if arm == 'withdrawn': return 0.0 if step > switch else LAMBDA
    raise ValueError(arm)

def free_ckpt(load, seed, switch=None):
    switch = SWITCH_OF.get(load, 3000) if switch is None else switch
    name = f"free_L{load}_s{seed}_sw{switch}_f{FORMAT_VERSION}"
    ck = ckpt_load(name)
    if ck is not None:
        return ck
    print(f"  [free hybrid load {load}] training to step {switch}", flush=True)
    set_seed(seed)
    model = build('hybrid', seed)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG.lr, betas=(0.9, 0.95), weight_decay=CFG.weight_decay)
    scaler = torch.amp.GradScaler('cuda', enabled=IS_CUDA and AMP_DTYPE == torch.float16)
    rng = np.random.default_rng([seed, load, 7, switch])
    gen = torch.Generator(device=DEVICE); gen.manual_seed(int(rng.integers(2**62)))
    batches = eval_batches(load)
    model.train()
    for step in range(1, switch + 1):
        for g in opt.param_groups: g['lr'] = CFG.lr * min(1.0, step / CFG.warmup)
        x_key, x_val, y = make_batch(CFG.batch, load, gen); qpos = qpos_of(y)
        with amp(): lg = model(x_key, x_val, qpos)[0]
        loss = F.cross_entropy(lg.float().reshape(-1, VOCAB), y[:, qpos].reshape(-1))
        opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip); scaler.step(opt); scaler.update()
        if step % 1000 == 0:
            ev = evaluate(model, batches)
            print(f"    step {step:>5} acc {ev['acc']:.3f} D_rec {ev['D_rec']:.2f} D_ret {ev['D_ret']:.2f}")
    ev = evaluate(model, batches)
    ck = {'model': model.state_dict(), 'opt': opt.state_dict(), 'scaler': scaler.state_dict(), 'eval': ev}
    ckpt_save(name, ck)
    del model, opt
    if IS_CUDA: torch.cuda.empty_cache()
    return ck

def steer_arm(load, arm, seed):
    switch = SWITCH_OF[load]
    from_scratch = arm in ('maintained', 'withdrawn')
    batches = eval_batches(load)
    set_seed(seed); model = build('hybrid', seed)
    opt = torch.optim.AdamW(model.parameters(), lr=CFG.lr, betas=(0.9, 0.95), weight_decay=CFG.weight_decay)
    scaler = torch.amp.GradScaler('cuda', enabled=IS_CUDA and AMP_DTYPE == torch.float16)
    start = 0
    if not from_scratch:
        ck = free_ckpt(load, seed, switch)
        model.load_state_dict(ck['model']); opt.load_state_dict(ck['opt'])
        if ck['scaler']: scaler.load_state_dict(ck['scaler'])
        start = switch
    total = switch + POST
    rng = np.random.default_rng([seed, load, sum(ord(x) for x in arm), start])
    gen = torch.Generator(device=DEVICE); gen.manual_seed(int(rng.integers(2**62)))
    trace, losses, prices = [], [], []
    ev = evaluate(model, batches)
    trace.append({'step': start, 'loss': float('nan'), 'price_term': float('nan'),
                  'lam': price_of(start, arm, switch), **ev})
    model.train(); sync(); t0 = time.time()
    for step in range(start + 1, total + 1):
        for g in opt.param_groups: g['lr'] = CFG.lr * min(1.0, step / CFG.warmup)
        lam = price_of(step, arm, switch)
        x_key, x_val, y = make_batch(CFG.batch, load, gen); qpos = qpos_of(y)
        with amp(): out, lh, lc = model(x_key, x_val, qpos)
        loss = F.cross_entropy(out.float().reshape(-1, VOCAB), y[:, qpos].reshape(-1))
        pen = lc.float().norm(dim=-1).mean()
        opt.zero_grad(set_to_none=True); scaler.scale(loss + lam * pen).backward(); scaler.unscale_(opt)
        nn.utils.clip_grad_norm_(model.parameters(), CFG.grad_clip); scaler.step(opt); scaler.update()
        losses.append(float(loss.detach())); prices.append(float(pen.detach()))
        if step % STEER_LOG == 0 or step == total:
            ev = evaluate(model, batches)
            trace.append({'step': step, 'loss': float(np.mean(losses)), 'price_term': float(np.mean(prices)),
                          'lam': lam, **ev})
            losses, prices = [], []; model.train()
    sync(); out = {**evaluate(model, batches), 'trace': trace, 'minutes': (time.time()-t0)/60}
    del model, opt
    if IS_CUDA: torch.cuda.empty_cache()
    return out

STEER=[]
for load in (STEER_POS, STEER_NEG):
    switch = SWITCH_OF[load]
    print(f"\n=== load {load}, switch {switch} ===")
    for seed in STEER_SEEDS:
        for arm in STEER_ARMS:
            spec={'block':'steer','load':load,'arm':arm,'seed':seed,'size':SIZE,'switch':switch,
                  'post':POST,'lam':LAMBDA,'cfg':asdict(CFG),'format':FORMAT_VERSION,'steer_format':FORMAT_STEER}
            r=REC.get(spec)
            if r is None: r=REC.put(spec, steer_arm(load,arm,seed))
            STEER.append({'load':load,'arm':arm,'seed':seed,**{k:v for k,v in r.items() if not k.startswith('_') and k!='trace'}})
ST=pd.DataFrame(STEER)
ST.to_csv(TABDIR/'stage2_arms_final.csv',index=False)
print(ST.groupby(['load','arm'])[['acc','D_rec','D_ret','acc_no_ret']].agg(['mean','std']).round(3))


## Price selection at load 8

In [ ]:
# What price actually moves the route? A short late-switch sweep at the positive load.
# Your paper picks lambda from a measured boundary; the toy model's 0.01 has no meaning at d=512.
SWITCH, SWEEP_POST, SWEEP_LOAD = 3000, 2000, 8
LAMBDAS = [0.0, 0.003, 0.01, 0.03, 0.1, 0.3, 1.0]

ck = free_ckpt(SWEEP_LOAD, 0, SWITCH)          # defined in the Stage 2 cell; trains once, then cached
print(f"free hybrid at step {SWITCH}: acc {ck['eval']['acc']:.3f}, "
      f"D_rec {ck['eval']['D_rec']:.2f}, D_ret {ck['eval']['D_ret']:.2f}\n")
batches = eval_batches(SWEEP_LOAD)
rows = []
for lam in LAMBDAS:
    spec = {"block": "lamsweep", "load": SWEEP_LOAD, "lam": lam, "switch": SWITCH,
            "post": SWEEP_POST, "size": SIZE, "cfg": asdict(CFG), "format": FORMAT_VERSION}
    r = REC.get(spec)
    if r is None:
        set_seed(0)
        m = build("hybrid", 0)
        m.load_state_dict(ck["model"])
        opt = torch.optim.AdamW(m.parameters(), lr=CFG.lr, betas=(0.9, 0.95),
                                weight_decay=CFG.weight_decay)
        opt.load_state_dict(ck["opt"])
        scaler = torch.amp.GradScaler("cuda", enabled=IS_CUDA and AMP_DTYPE == torch.float16)
        if ck["scaler"]:
            scaler.load_state_dict(ck["scaler"])
        gen = torch.Generator(device=DEVICE); gen.manual_seed(1234)
        m.train()
        for step in range(SWITCH + 1, SWITCH + SWEEP_POST + 1):
            x_key, x_val, y = make_batch(CFG.batch, SWEEP_LOAD, gen)
            qpos = qpos_of(y)
            with amp():
                out, lh, lc = m(x_key, x_val, qpos)
            loss = F.cross_entropy(out.float().reshape(-1, VOCAB), y[:, qpos].reshape(-1))
            pen = lc.float().norm(dim=-1).mean()
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss + lam * pen).backward()
            scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(m.parameters(), CFG.grad_clip)
            scaler.step(opt); scaler.update()
        r = REC.put(spec, {**evaluate(m, batches), "price_term": float(pen.detach())})
        del m, opt
        if IS_CUDA:
            torch.cuda.empty_cache()
    rows.append({"lam": lam, **{k: r[k] for k in ("acc", "D_rec", "D_ret", "acc_no_ret", "price_term")}})
    print(f"lambda {lam:<6} acc {r['acc']:.3f}  D_rec {r['D_rec']:.2f}  D_ret {r['D_ret']:.2f}  "
          f"||l_c|| {r['price_term']:.3f}  accNoRet {r['acc_no_ret']:.3f}", flush=True)

SW = pd.DataFrame(rows)
SW.to_csv(TABDIR / "lambda_sweep.csv", index=False)
ok = SW[(SW.D_rec > SW.D_ret) & (SW.acc >= 0.95 * SW.loc[SW.lam == 0, "acc"].item())]
print(f"\ncrossing with accuracy preserved at: {list(ok.lam) or 'none'}")
print("pick the smallest such lambda for Stage 2; if none, the sweep needs to extend higher")

## Figures and paper tables from stored records

In [ ]:
# ============ FIGURES AND TABLES FOR THE SCALE SECTION ============
# Paste as a NEW cell and run. Reads everything from records.jsonl; trains nothing. Produces:
#   fig_scale_arms.pdf       four-arm dependence trajectories, both loads (main-text candidate)
#   fig_scale_boundary.pdf   viability screen + lambda dose-response (appendix)
#   tab_scale_arms.tex       the four-arm table, 3 seeds, mean +/- sd
#   tab_scale_screen.tex     the viability screen
#   scale_numbers.txt        every number quoted in the draft text, for checking
import itertools

SWITCH_OF = {8: 3000, 32: 5000}          # set per load from when the free hybrid converges
LOADS_FIG = [8, 32]
ARMS_FIG = ["never", "late", "maintained", "withdrawn"]
COL = {"never": "#1b6ca8", "late": "#e07a1f", "maintained": "#3a9d5d", "withdrawn": "#d1495b"}
LBL = {"never": "never", "late": "late", "maintained": "maintained", "withdrawn": "withdrawn"}


def recs(**match):
    out = []
    for r in REC.recs.values():
        s = r["_spec"]
        if all(s.get(k) == v for k, v in match.items()):
            out.append(r)
    return out


def arm_records(load, arm):
    return [r for r in recs(block="steer", load=load, arm=arm, format=FORMAT_VERSION,
                            lam=LAMBDA, post=POST, switch=SWITCH_OF[load])]


def agg(load, arm, key):
    v = [r[key] for r in arm_records(load, arm)]
    return (float(np.mean(v)), float(np.std(v)), len(v)) if v else (np.nan, np.nan, 0)


def traces(load, arm):
    return [pd.DataFrame(r["trace"]) for r in arm_records(load, arm)]


# ---------- figure 1: four-arm trajectories ----------
mpl.rcParams.update({"figure.dpi": 130, "savefig.dpi": 300, "savefig.bbox": "tight", "font.size": 8,
                     "axes.labelsize": 8, "axes.titlesize": 8.5, "legend.fontsize": 7,
                     "xtick.labelsize": 7, "ytick.labelsize": 7,
                     "axes.spines.top": False, "axes.spines.right": False})
fig, axes = plt.subplots(2, len(LOADS_FIG), figsize=(3.3 * len(LOADS_FIG), 4.0), sharey="row")
for j, load in enumerate(LOADS_FIG):
    sw = SWITCH_OF[load]
    for i, (key, name) in enumerate([("D_rec", "recurrent dependence $D_h$"),
                                     ("D_ret", "retrieval dependence $D_c$")]):
        ax = axes[i, j]
        for arm in ARMS_FIG:
            ts = traces(load, arm)
            if not ts:
                continue
            g = pd.concat(ts).groupby("step")[key].agg(["mean", "std"]).reset_index()
            sd = g["std"].fillna(0)
            ax.plot(g.step, g["mean"], "-", lw=1.2, color=COL[arm], label=LBL[arm])
            ax.fill_between(g.step, g["mean"] - sd, g["mean"] + sd, color=COL[arm], alpha=0.15, lw=0)
        ax.axvline(sw, color="0.55", lw=0.7, ls=":")
        ax.set_ylim(-0.03, 1.05)
        if i == 0:
            ax.set_title(f"load {load}" + (" (recurrence viable)" if load == 8 else " (not viable)"))
        if j == 0:
            ax.set_ylabel(name)
        if i == 1:
            ax.set_xlabel("training step")
h, l = axes[0, 0].get_legend_handles_labels()
fig.tight_layout()
fig.legend(h, l, loc="upper center", ncol=4, frameon=False, bbox_to_anchor=(0.5, 0.0))
for ext in ("pdf", "png"):
    fig.savefig(FIGDIR / f"fig_scale_arms.{ext}")
plt.show()

# ---------- figure 2: viability screen and lambda dose-response ----------
scr = sorted([(r["_spec"]["load"], r["_spec"]["kind"], r["acc"])
              for r in recs(block="run", format=FORMAT_VERSION, steps=STEPS)])
S = pd.DataFrame(scr, columns=["load", "kind", "acc"]).pivot_table(index="load", columns="kind", values="acc")
sweep = sorted([(r["_spec"]["lam"], r["D_rec"], r["D_ret"], r["acc"])
                for r in recs(block="lamsweep", format=FORMAT_VERSION)])
W = pd.DataFrame(sweep, columns=["lam", "D_rec", "D_ret", "acc"])

fig2, ax2 = plt.subplots(1, 2, figsize=(6.6, 2.5))
for kind, c, lab in [("recurrent_only", "#1b6ca8", "recurrence alone"),
                     ("hybrid", "#3a9d5d", "free hybrid")]:
    if kind in S:
        ax2[0].plot(S.index, S[kind], "-o", ms=3.5, color=c, label=lab)
ax2[0].axhline(0.9, color="0.55", lw=0.7, ls="--")
ax2[0].set_xscale("log", base=2); ax2[0].set_xticks(list(S.index)); ax2[0].set_xticklabels(list(S.index))
ax2[0].set_xlabel("memory load $L$"); ax2[0].set_ylabel("accuracy"); ax2[0].set_ylim(-0.03, 1.05)
ax2[0].set_title("route viability"); ax2[0].legend(frameon=False, loc="lower left")
if len(W):
    ax2[1].plot(W.lam, W.D_rec, "-o", ms=3.5, color="#1b6ca8", label="$D_h$")
    ax2[1].plot(W.lam, W.D_ret, "-o", ms=3.5, color="#e07a1f", label="$D_c$")
    ax2[1].plot(W.lam, W.acc, ":s", ms=3, color="0.35", label="accuracy")
    ax2[1].set_xscale("symlog", linthresh=1e-3)
    ax2[1].set_xlabel(r"retrieval price $\lambda$"); ax2[1].set_ylim(-0.03, 1.05)
    ax2[1].set_title("dose-response at load 8"); ax2[1].legend(frameon=False, loc="center right")
fig2.tight_layout()
for ext in ("pdf", "png"):
    fig2.savefig(FIGDIR / f"fig_scale_boundary.{ext}")
plt.show()

# ---------- tables ----------
def cell(load, arm, key, fmt="{:.3f}"):
    m, s, n = agg(load, arm, key)
    return "--" if n == 0 else (fmt.format(m) + (f" $\\pm$ {fmt.format(s)}" if n > 1 else ""))


def crossings(load, arm):
    rs = arm_records(load, arm)
    return f"{sum(1 for r in rs if r['D_rec'] > r['D_ret'])}/{len(rs)}" if rs else "--"


n_par = sum(p.numel() for p in build("hybrid", 0).parameters())
tex = [r"\begin{table}[t]", r"\centering", r"\small",
       rf"\caption{{\textbf{{Post-convergence steering in the large-scale selective-recall model.}} Four arms at a load "
       rf"where recurrence is independently viable (L=8, attention-free accuracy 0.994) and at a boundary-selected "
       rf"control where it is not (L=32, 0.251). Same price $\lambda={LAMBDA}$ on $\mathbb{{E}}\|\ell_c\|$; the "
       rf"switch step is set per load from the free model's convergence (3000 and 5000). Mean $\pm$ s.d. over "
       rf"three seeds; ``inverted'' counts seeds with $D_h > D_c$ at the end.}}",
       r"\label{tab:scale}", r"\begin{tabular}{llccccc}", r"\toprule",
       r"load & arm & accuracy & $D_h$ & $D_c$ & acc. w/o retrieval & inverted \\", r"\midrule"]
for load in LOADS_FIG:
    for k, arm in enumerate(ARMS_FIG):
        first = rf"\multirow{{4}}{{*}}{{{load}}}" if k == 0 else ""
        tex.append(" & ".join([first, arm, cell(load, arm, "acc"), cell(load, arm, "D_rec", "{:.2f}"),
                               cell(load, arm, "D_ret", "{:.2f}"), cell(load, arm, "acc_no_ret"),
                               crossings(load, arm)]) + r" \\")
    if load != LOADS_FIG[-1]:
        tex.append(r"\midrule")
tex += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
(TABDIR / "tab_scale_arms.tex").write_text("\n".join(tex))

tex2 = [r"\begin{table}[t]", r"\centering", r"\small",
        r"\caption{\textbf{Route viability and the free hybrid's choice}, measured before any steering. "
        r"Single-route models are trained from scratch with the same budget and capacity.}",
        r"\label{tab:scale-screen}", r"\begin{tabular}{rccc}", r"\toprule",
        r"load & recurrence alone & free hybrid acc. & free hybrid $D_h$ / $D_c$ \\", r"\midrule"]
for L in S.index:
    hyb = [r for r in recs(block="run", kind="hybrid", load=int(L), format=FORMAT_VERSION, steps=STEPS)]
    d = f"{hyb[0]['D_rec']:.2f} / {hyb[0]['D_ret']:.2f}" if hyb else "--"
    tex2.append(f"{int(L)} & {S.loc[L, 'recurrent_only']:.3f} & "
                f"{S.loc[L, 'hybrid']:.3f} & {d} " + r"\\")
tex2 += [r"\bottomrule", r"\end{tabular}", r"\end{table}"]
(TABDIR / "tab_scale_screen.tex").write_text("\n".join(tex2))

# ---------- every number quoted in the draft ----------
out = [f"parameters: {n_par / 1e6:.2f}M (d={CFG.d}, {CFG.n_layers} layers, {CFG.n_heads} heads)",
       f"task: selective recall, {N_ITEMS} symbols, chance {CHANCE:.4f}; {STEPS} steps, batch {CFG.batch}",
       f"price: lambda={LAMBDA} on E||l_c||; post-switch steps {POST}", "", "viability screen:"]
for L in S.index:
    out.append(f"  load {int(L):>2}: recurrence alone {S.loc[L, 'recurrent_only']:.3f}, "
               f"free hybrid {S.loc[L, 'hybrid']:.3f}")
out += ["", "lambda sweep at load 8 (2000 post-switch steps):"]
for _, r in W.iterrows():
    out.append(f"  lambda {r.lam:<7} D_h {r.D_rec:.2f}  D_c {r.D_ret:.2f}  acc {r.acc:.3f}")
out += ["", "four arms (mean +/- sd over 3 seeds):"]
for load in LOADS_FIG:
    out.append(f"  -- load {load}, switch {SWITCH_OF[load]} --")
    for arm in ARMS_FIG:
        rs = arm_records(load, arm)
        if not rs:
            continue
        a, sa, n = agg(load, arm, "acc")
        h, sh, _ = agg(load, arm, "D_rec")
        c, sc, _ = agg(load, arm, "D_ret")
        nr, snr, _ = agg(load, arm, "acc_no_ret")
        out.append(f"    {arm:<11} acc {a:.3f}+/-{sa:.3f}  D_h {h:.2f}+/-{sh:.2f}  D_c {c:.2f}+/-{sc:.2f}  "
                   f"accNoRet {nr:.3f}+/-{snr:.3f}  inverted {crossings(load, arm)}  n={n}")
        out.append(f"      per-seed D_h {[round(r['D_rec'], 2) for r in rs]}, "
                   f"D_c {[round(r['D_ret'], 2) for r in rs]}, acc {[round(r['acc'], 3) for r in rs]}")
for load in LOADS_FIG:
    for arm in ("late",):
        ts = traces(load, arm)
        if ts:
            mins = [float(t[t.step > SWITCH_OF[load]].acc.min()) for t in ts]
            out.append(f"  post-switch minimum accuracy, {arm} load {load}: "
                       f"mean {np.mean(mins):.3f}, worst {min(mins):.3f}")
(TABDIR / "scale_numbers.txt").write_text("\n".join(out))
print("\n".join(out))
print(f"\nwrote {FIGDIR}/fig_scale_arms.pdf, fig_scale_boundary.pdf and "
      f"{TABDIR}/tab_scale_arms.tex, tab_scale_screen.tex, scale_numbers.txt")